# **Procesamiento del lenguaje natural**

**Tabajo parcial II**

**Tema:** Agente Conversacional para Admisión y Nivelación de la UG

Integrantes:


*   Caicho Almeida Shanney
*   Cedeño Pinto Scarlett



In [2]:
# Importacion de librerias
!pip install nltk
!pip install git+https://github.com
import unicodedata
import re
import nltk
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
import pandas as pd
nltk.download('stopwords', quiet=True)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import random
from sklearn.metrics import (accuracy_score,f1_score,classification_report)
import gradio as gr

  Cloning https://github.com to /tmp/pip-req-build-v4t1zezd
  Running command git clone --filter=blob:none --quiet https://github.com /tmp/pip-req-build-v4t1zezd
  fatal: repository 'https://github.com/' not found
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com /tmp/pip-req-build-v4t1zezd did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com /tmp/pip-req-build-v4t1zezd did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


**Carga de datos**

In [3]:
# Ruta del JSON
url = "https://raw.githubusercontent.com/Shaoss3/PLN_Agente_Conversacional/refs/heads/main/data/intenciones.json"

url_corpus = "https://raw.githubusercontent.com/Shaoss3/PLN_Agente_Conversacional/refs/heads/main/data/corpus_nivelacion.json"
# Definición del umbral
UMBRAL = 0.25 # Umbral regular para chatbots cerrados aplica fallback

In [4]:
# Listado de referencias
CARRERAS = [
  "arquitectura", "ciencia de datos e inteligencia artificial", "ingenieria civil",
  "ingenieria Industrial", "mecatronica", "sistemas de informacion", "software",
  "tecnologias de la informacion", "telematica", "ingenieria de la produccion",
  "medicina", "odontologia"
]

TERMINOS_PROCESO = [
    "cupo aceptado", "curso de nivelacion", "registro nacional",
    "siug", "matricula", "postulacion", "evaluacion de conocimientos"
]

**Preprocesamiento de texto**

In [5]:
def preprocesar (texto):
  """
    Transforma el texto  crudo para que el TF-IDF pueda vecctorizar
    de forma correcta

    """
  # Conversion a minúsculas
  texto = texto.lower()

  # Quitar tildes - acentos
  texto = unicodedata.normalize('NFKD', texto)
  texto = texto.encode('ascii', 'ignore').decode('ascii')

  # Quitar puntuacion y caracteres especiales
  texto = re.sub(r'[^a-zA-Z\s]', '', texto)

  # Tokenizacion
  tokens = texto.split()

  # Eliminacion de stopwords
  stop = set(stopwords.words('spanish'))
  tokens = [palabra for palabra in tokens if palabra not in stop]

  # Stemming
  stemmer = SnowballStemmer('spanish')
  tokens = [stemmer.stem(t) for t in tokens]

  return ' '.join(tokens)

**Extraccion de entidades**

In [6]:
def extraer_entidades(texto):
    """
    Detecta entidades relevantes en la consulta del usuario.
    Recibe texto crudo (sin preprocesar).

    """
    texto_lower = texto.lower()
    entidades = {}

    # Carrera mencionada
    for carrera in CARRERAS:
        if carrera in texto_lower:
            entidades['carrera'] = carrera
            break

    # Término específico del proceso
    for termino in TERMINOS_PROCESO:
        if termino in texto_lower:
            entidades['termino_proceso'] = termino
            break

    return entidades

In [7]:
def cargar_datos(url):

  """
  Carga de las intenciones.json

  """

  # Carga de los datos a un DataFRame
  intenciones_datos = pd.read_json(url)

  etiquetas  = []   # tag por cada utterance
  utterances = []   # texto preprocesado de cada frase
  respuestas = {}   # diccionario tag → lista de respuestas


  for intencion in intenciones_datos['intenciones']:
      tag = intencion['tag']
      respuestas[tag] = intencion['respuestas']
      for frase in intencion['utterances']:
          utterances.append(preprocesar(frase))
          etiquetas.append(tag)          # ← etiquetas, no tag

  return utterances, etiquetas, respuestas

In [8]:
def entrenar(utterances):
    """Ajusta el vectorizador TF-IDF sobre todos los utterances."""

    # Aplicacion función TF-IDF
    vectorizador = TfidfVectorizer(ngram_range=(1,2)) # bigrama y unigrama
    matriz = vectorizador.fit_transform(utterances)
    return vectorizador, matriz

**Detección de intensiones (TF-IDF + Similitud del coseno)**

In [9]:
def detectar_intencion(consulta, vectorizador, matriz, etiquetas):
    """
    Preprocesa la consulta, la vectoriza y calcula similitud coseno
    contra todos los utterances. Devuelve (tag, similitud).
    """

    #Simlitud coseno
    consulta_limpia  = preprocesar(consulta)
    vector_consulta  = vectorizador.transform([consulta_limpia])
    similitudes      = cosine_similarity(vector_consulta, matriz)[0]
    idx_mejor        = similitudes.argmax()
    return etiquetas[idx_mejor], similitudes[idx_mejor]

In [10]:
def obtener_respuesta(tag, respuestas):
    """Devuelve una respuesta aleatoria del tag indicado."""
    return random.choice(respuestas[tag])

In [11]:
def cargar_corpus(url_corpus):
    """
    Carga corpus.json desde URL y construye vectorizador TF-IDF
    sobre el contenido de cada sección.
    """
    datos = pd.read_json(url_corpus)

    tags_corpus      = []
    textos_originales = []
    contenidos_limpios = []

    for seccion in datos['secciones']:
        tags_corpus.append(seccion['tag'])
        textos_originales.append(seccion['contenido'])
        contenidos_limpios.append(preprocesar(seccion['contenido']))

    vectorizador_corpus = TfidfVectorizer(ngram_range=(1, 2))
    matriz_corpus = vectorizador_corpus.fit_transform(contenidos_limpios)

    return vectorizador_corpus, matriz_corpus, tags_corpus, textos_originales


In [12]:
def buscar_en_corpus(consulta, tag_detectado,
                     vectorizador_corpus, matriz_corpus,
                     tags_corpus, textos_originales,
                     umbral_corpus=0.10):
    """
    Busca el fragmento más relevante del corpus para la consulta.
    Prioriza la sección cuyo tag coincida con la intención detectada.
    """
    consulta_limpia = preprocesar(consulta)
    vector = vectorizador_corpus.transform([consulta_limpia])
    similitudes = cosine_similarity(vector, matriz_corpus)[0]

    # Boost a la sección que coincide con la intención detectada
    for i, tag in enumerate(tags_corpus):
        if tag == tag_detectado:
            similitudes[i] *= 1.5
            break

    idx_mejor = similitudes.argmax()
    sim_mejor = similitudes[idx_mejor]

    if sim_mejor >= umbral_corpus:
        return textos_originales[idx_mejor]
    return None

In [14]:
def responder_umbral(consulta, vectorizador, matriz, etiquetas, respuestas):
    """
    Detecta intención, aplica umbral y devuelve respuesta.

    """

    RESPUESTA_FALLBACK = (

        "Lo siento, no entendí tu consulta. "
        "¿Puedes reformularla? Solo puedo responder preguntas sobre "
        "admisión y nivelación de la Universidad de Guayaquil."
    )

    tag, similitud = detectar_intencion(consulta, vectorizador, matriz, etiquetas)
    entidades = extraer_entidades(consulta)

    if similitud >= UMBRAL:
        respuesta = random.choice(respuestas[tag])
    else:
        tag = 'fallback'
        respuesta = RESPUESTA_FALLBACK

    return {
        'tag'       : tag,
        'similitud' : round(float(similitud), 4),
        'respuesta' : respuesta,
        'entidades' : entidades
    }

In [15]:
def responder_con_corpus(consulta, vectorizador, matriz, etiquetas,
                         respuestas, vectorizador_corpus, matriz_corpus,
                         tags_corpus, textos_originales):
    """
    Detecta intención y enriquece la respuesta con información del corpus.
    """
    tag, similitud = detectar_intencion(
        consulta, vectorizador, matriz, etiquetas
    )
    entidades = extraer_entidades(consulta)

    if similitud >= UMBRAL:
        respuesta_base = random.choice(respuestas[tag])

        # Buscar información adicional en el corpus
        fragmento = buscar_en_corpus(
            consulta, tag,
            vectorizador_corpus, matriz_corpus,
            tags_corpus, textos_originales
        )

        if fragmento:
            respuesta_final = (
                f"{respuesta_base}\n\n"
                f"Información adicional:\n{fragmento}"
            )
        else:
            respuesta_final = respuesta_base

        # Personalizar si hay carrera detectada
        if entidades.get('carrera'):
            respuesta_final += (
                f"\n\nCarrera detectada: "
                f"{entidades['carrera'].title()}"
            )
    else:
        tag = 'fallback'
        respuesta_final = (
            "Lo siento, no entendí tu consulta. "
            "¿Puedes reformularla? Solo puedo responder preguntas "
            "sobre admisión y nivelación de la Universidad de Guayaquil."
        )

    return {
        'tag'      : tag,
        'similitud': round(float(similitud), 4),
        'respuesta': respuesta_final,
        'entidades': entidades
    }

In [16]:
#Vista de la matriz ItF-IDF
utterances, etiquetas, respuestas = cargar_datos(url)
vectorizador, matriz = entrenar(utterances)

# Ver dimensiones
print(f"Utterances: {matriz.shape[0]}")   # filas = total utterances
print(f"Vocabulario: {matriz.shape[1]}")  # columnas = palabras únicas

# Ver la matriz como DataFrame
df_matriz = pd.DataFrame(
    matriz.toarray(),
    columns=vectorizador.get_feature_names_out(),
    index=etiquetas
)
print(df_matriz)

Utterances: 112
Vocabulario: 269
                     abren  abren inscripcion  acced  acced clas  adi  \
saludo                 0.0                0.0    0.0         0.0  0.0   
saludo                 0.0                0.0    0.0         0.0  0.0   
saludo                 0.0                0.0    0.0         0.0  0.0   
saludo                 0.0                0.0    0.0         0.0  0.0   
saludo                 0.0                0.0    0.0         0.0  0.0   
...                    ...                ...    ...         ...  ...   
informacion_carrera    0.0                0.0    0.0         0.0  0.0   
informacion_carrera    0.0                0.0    0.0         0.0  0.0   
informacion_carrera    0.0                0.0    0.0         0.0  0.0   
informacion_carrera    0.0                0.0    0.0         0.0  0.0   
informacion_carrera    0.0                0.0    0.0         0.0  0.0   

                     admision  admision contien  admision ug  \
saludo                    

**Conjunto de prueba**

In [17]:
conjunto_prueba = [
    # Consultaas / Intenciones esperadas
    (" como puedo registrarme en la ug", "proceso_admision"),
    (" q documentos necesito para poder inscribirme a inegieria de la produccion", "requisitos_admision"),
    (" cuando son las fechas de inscripsion", "fechas_registro"), # error tipografico
    ("como se crea una cuenta en el siug", "crear_cuenta_siug"),
    ("Con cuanto apruebo nivelasion", "aprobacion_nivelacion"), # error tipografico
    ("que toman en el examen de ingreso", "aplicacion_evaluaciones"),
    ("como me asignan el cupo", "asignacion_cupo"),
    ("cuando son las clases", "clases_nivelacion"),
    ("como me matriculo", "matricula_nivelacion"),
    ("quien gano el mundial", "fuera_dominio"),  # fuera de dominio
    ("como se si aprobe nibelacion", "aprobacion_nivelacion"), # error tipografico
    ("buenas tardes",                                "saludo"),
    ("hola necesito ayuda con admision",             "saludo"),
    ("muchas gracias eso era todo",                  "despedida"),
    ("ok ya entendi chao",                           "despedida"),
    ("que documentos pido para entrar a la ug",      "requisitos_admision"),
    ("en q fecha me registro segun mi cedula",       "fechas_registro"),
    ("no tengo usuario en el siug q hago",           "crear_cuenta_siug"),
    ("con q puntaje me dan el cupo",                 "asignacion_cupo"),
    ("como entro a zoom para mis clases",            "clases_nivelacion"),
    ("donde hago la matricula del curso",            "matricula_nivelacion"),
    ("como se hace una torta",                       "fuera_dominio"),
    ("recomiendame una pelicula",                    "fuera_dominio"),
    ("cuales son los pasos para entrar a la ug",     "proceso_admision"),
    ("como es la prueba de ingreso",                 "aplicacion_evaluaciones"),
]

Evaluacion de agente - Métricas de evaluación

In [18]:
def evaluar_agente (conjunto_prueba, vectorizador, matriz, etiquetas, respuestas):
    """
    Corre responder_umbral sobre cada consulta del conjunto de prueba.
    Compara la intención predicha con la esperada y calcula métricas.
    """
    reales     = []  # intenciones esperadas
    predichas  = []  # intenciones que predijo el agente

    for consulta, intencion_real in conjunto_prueba:
        resultado = responder_umbral(
            consulta, vectorizador, matriz, etiquetas, respuestas
        )
        reales.append(intencion_real)
        predichas.append(resultado['tag'])

    # Métricas
    acc = accuracy_score(reales, predichas)
    f1  = f1_score(reales, predichas, average='macro', zero_division=0)

    print(f"Accuracy : {acc:.2%}")
    print(f"F1-macro : {f1:.2%}")
    print()
    print("Reporte por intención:")
    print(classification_report(reales, predichas, zero_division=0))

    #  Tabla de resultados
    df = pd.DataFrame({
        'consulta'  : [c for c, _ in conjunto_prueba],
        'esperada'  : reales,
        'predicha'  : predichas,
        'correcta'  : [r == p for r, p in zip(reales, predichas)]
    })
    print(df.to_string(index=False))
    return df

In [19]:
df_resultados = evaluar_agente(
    conjunto_prueba, vectorizador, matriz, etiquetas, respuestas
)

Accuracy : 88.00%
F1-macro : 79.41%

Reporte por intención:
                         precision    recall  f1-score   support

aplicacion_evaluaciones       1.00      1.00      1.00         2
  aprobacion_nivelacion       1.00      1.00      1.00         2
        asignacion_cupo       1.00      1.00      1.00         2
      clases_nivelacion       1.00      1.00      1.00         2
      crear_cuenta_siug       1.00      0.50      0.67         2
              despedida       1.00      1.00      1.00         2
        fechas_registro       1.00      1.00      1.00         2
          fuera_dominio       0.75      1.00      0.86         3
   matricula_nivelacion       1.00      1.00      1.00         2
       oferta_academica       0.00      0.00      0.00         0
       proceso_admision       0.00      0.00      0.00         2
    requisitos_admision       0.67      1.00      0.80         2
                 saludo       1.00      1.00      1.00         2

               accuracy     

**Manejo de diálogo y respuesta de fallback**

In [20]:
# Ver solo las predicciones incorrectas
incorrectas = df_resultados[df_resultados['correcta'] == False]
print(incorrectas[['consulta', 'esperada', 'predicha']].to_string(index=False))

#mejorar utterances en el JSON

                                consulta          esperada            predicha
         como puedo registrarme en la ug  proceso_admision    oferta_academica
      no tengo usuario en el siug q hago crear_cuenta_siug       fuera_dominio
cuales son los pasos para entrar a la ug  proceso_admision requisitos_admision


**Aplicación Web**

In [21]:


# Cargar intenciones y corpus
utterances, etiquetas, respuestas = cargar_datos(url)
vectorizador, matriz = entrenar(utterances)
vectorizador_corpus, matriz_corpus, tags_corpus, textos = cargar_corpus(url_corpus)

print("Agente con corpus cargado correctamente.")

# Interfaz Gradio
def chat(mensaje, historial):
    resultado = responder_con_corpus(
        mensaje, vectorizador, matriz, etiquetas, respuestas,
        vectorizador_corpus, matriz_corpus, tags_corpus, textos
    )
    return resultado['respuesta']

interfaz = gr.ChatInterface(
    fn=chat,
    title="Agente Conversacional - Admisión y Nivelación UG",
    description="Consulta información sobre admisión y nivelación de la Universidad de Guayaquil.",
    examples=[
        "¿Cuánto dura la carrera de medicina?",
        "¿Cómo me matriculo en nivelación?",
        "¿Con cuánto se aprueba nivelación?",
        "¿Qué carreras ofrece la UG?",
    ]
)

interfaz.launch()

Agente con corpus cargado correctamente.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa5ce5afee84a50105.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
